# 10: Profile & Branding System Testing

This notebook tests the complete Profile/Branding workflow:

1. **User Profile** - Create and persist user configuration
2. **Client Profile** - Create Hillcrest as a client with branding
3. **1Password Integration** - Get credentials from 1Password
4. **Branded Report** - Generate a PDF with client branding
5. **Secure Shell Execution** - Validate the security-hardened subprocess layer

## Prerequisites

- 1Password CLI (`op`) installed and authenticated
- Run on macOS for Keychain support

In [ ]:
import os
import sys
from pathlib import Path
from datetime import datetime

import siege_utilities as su
su.configure_shared_logging(level="INFO")

# Ensure siege_utilities is importable
sys.path.insert(0, str(Path.cwd().parent))

su.log_info("=== System Check ===")
su.log_info(f"Python: {sys.version}")
su.log_info(f"Working Directory: {Path.cwd()}")
su.log_info(f"Platform: {sys.platform}")

# --- Output configuration ---
OUTPUT_DIR = Path("output") / "notebook_10"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

su.log_info(f"Output directory: {OUTPUT_DIR}  (exists={OUTPUT_DIR.exists()})")

## 1. Check Credential Backends

Check which credential storage backends are available.

In [ ]:
from siege_utilities.config import CredentialManager, credential_status

# Check backend status
su.log_info("=== Credential Backend Status ===")
status = credential_status()

for backend, info in status.items():
    symbol = "[OK]" if info['available'] else "[MISSING]"
    su.log_info(f"{symbol} {backend}: {info['status']}")
    su.log_info(f"   {info['description']}")

In [ ]:
# Initialize credential manager
cred_manager = CredentialManager()

su.log_info("=== Credential Manager Initialized ===")
su.log_info(f"Backend priority: {cred_manager.backend_priority}")
su.log_info(f"Available backends: {[k for k,v in cred_manager.available_backends.items() if v]}")
su.log_info(f"Default vault: {cred_manager.default_vault}")

## 2. Create User Profile

Create or load user profile with default preferences.

In [ ]:
# Modern Person/Actor imports
from siege_utilities.config import User, Client, BrandingConfig, ReportPreferences, HydraConfigManager

# Initialize HydraConfigManager (kept open for the notebook session)
manager = HydraConfigManager()

# Set up profiles directories
profiles_dir = Path.home() / ".siege_utilities" / "profiles"
users_dir = profiles_dir / "users"
clients_dir = profiles_dir / "clients"

su.log_info("=== Profile Directories ===")
su.log_info(f"Users: {users_dir} (exists: {users_dir.exists()})")
su.log_info(f"Clients: {clients_dir} (exists: {clients_dir.exists()})")

if users_dir.exists():
    existing_users = list(users_dir.glob("*.yaml"))
    su.log_info(f"Existing user profiles: {[f.stem for f in existing_users]}")

if clients_dir.exists():
    existing_clients = list(clients_dir.glob("*.yaml"))
    su.log_info(f"Existing client profiles: {[f.stem for f in existing_clients]}")

In [ ]:
# Create user using modern User model
user = User(
    person_id="dheeraj",
    name="Dheeraj Chand",
    email="dheeraj@siegeanalytics.com",
    username="dheeraj",
    github_login="dheerajchand",
    role="admin",
    preferred_download_directory=str(Path.home() / "Downloads" / "siege_utilities"),
    default_output_format="pdf",
    preferred_map_style="carto-positron",
    default_color_scheme="viridis",
    default_dpi=300,
    default_figure_size=(10, 8),
    enable_logging=True,
    log_level="INFO"
)

su.log_info("=== User Created ===")
su.log_info(f"Person ID: {user.person_id}")
su.log_info(f"Name: {user.name}")
su.log_info(f"Username: {user.username}")
su.log_info(f"Default output: {user.default_output_format}")

In [ ]:
# Save user profile using modern HydraConfigManager
success = manager.save_user(user, profiles_dir=users_dir)
su.log_info(f"User profile saved: {success}")

# Verify by loading
loaded_user = manager.load_user("dheeraj", profiles_dir=users_dir)
if loaded_user:
    su.log_info(f"Verified: Loaded user {loaded_user.person_id} ({loaded_user.name})")
else:
    su.log_warning("Could not load saved user profile")

## 3. Create Hillcrest Client Profile

Create a complete client profile with branding configuration.

In [ ]:
# Create Hillcrest branding config (typed BrandingConfig model)
# (Update these colors to match Hillcrest's actual branding)
hillcrest_branding = BrandingConfig(
    # Brand colors - UPDATE THESE with Hillcrest's actual colors
    primary_color="#1a4d2e",      # Forest green (placeholder)
    secondary_color="#4a7c59",    # Sage green (placeholder)
    accent_color="#f4a261",       # Warm accent (placeholder)
    text_color="#2d3436",         # Dark gray
    background_color="#ffffff",   # White
    
    # Fonts
    primary_font="Helvetica",
    secondary_font="Georgia",
    
    # Logo (update with actual path when available)
    logo_path=None,  # Will be set when logo is available
    
    # Layout
    header_height=50,
    footer_height=25,
    margin_top=25,
    margin_bottom=25,
    margin_left=20,
    margin_right=20,
    
    # Typography
    title_font_size=28,
    subtitle_font_size=20,
    body_font_size=12,
    caption_font_size=10,
    
    # Charts
    chart_color_palette="viridis",
    chart_background_color="#ffffff",
    chart_grid_color="#e0e0e0"
)

su.log_info("=== Hillcrest Branding Config ===")
su.log_info(f"Primary color: {hillcrest_branding.primary_color}")
su.log_info(f"Secondary color: {hillcrest_branding.secondary_color}")
su.log_info(f"Fonts: {hillcrest_branding.primary_font} / {hillcrest_branding.secondary_font}")
su.log_info(f"Color scheme: {hillcrest_branding.get_color_scheme()}")

In [ ]:
# Create Hillcrest report preferences
hillcrest_reports = ReportPreferences(
    default_format="pdf",
    include_executive_summary=True,
    chart_style="professional",
    page_size="Letter",
    orientation="landscape",
    include_table_of_contents=True,
    include_page_numbers=True,
    chart_quality="high",
    chart_dpi=300,
    include_chart_titles=True,
    include_chart_legends=True,
    sections=["executive_summary", "methodology", "findings", "recommendations"]
)

su.log_info("=== Hillcrest Report Preferences ===")
su.log_info(f"Format: {hillcrest_reports.default_format}")
su.log_info(f"Page: {hillcrest_reports.page_size} {hillcrest_reports.orientation}")
su.log_info(f"Sections: {hillcrest_reports.sections}")

In [ ]:
# Hillcrest contact info — in the modern Client model, these are direct fields
# (email, phone, website on the Person base class)
hillcrest_email = "contact@hillcrest.example.com"   # UPDATE with real email
hillcrest_phone = "(555) 123-4567"                   # UPDATE with real phone
hillcrest_website = "https://www.hillcrest.example.com"  # UPDATE with real website

su.log_info("=== Hillcrest Contact Info ===")
su.log_info(f"Email: {hillcrest_email}")
su.log_info(f"Phone: {hillcrest_phone}")
su.log_info(f"Website: {hillcrest_website}")

In [ ]:
# Create the complete Hillcrest client using modern Client model
hillcrest = Client(
    # Core Identity (from Person base)
    person_id="hillcrest",
    name="Hillcrest",
    email=hillcrest_email,
    phone=hillcrest_phone,
    website=hillcrest_website,
    
    # Client-specific fields
    client_code="HILL",
    industry="Political Consulting",  # UPDATE if different
    project_count=0,
    client_status="active",
    
    # Typed configuration models
    branding_config=hillcrest_branding,
    report_preferences=hillcrest_reports,
    notes="Primary client for Siege Analytics"
)

# Assign user to client (bidirectional)
hillcrest.assign_user(user.person_id)
hillcrest.set_primary_user(user.person_id)
user.assign_client(hillcrest.client_code)
user.set_primary_client(hillcrest.client_code)

su.log_info("=== Hillcrest Client Created ===")
su.log_info(f"Person ID: {hillcrest.person_id}")
su.log_info(f"Name: {hillcrest.name}")
su.log_info(f"Client Code: {hillcrest.client_code}")
su.log_info(f"Industry: {hillcrest.industry}")
su.log_info(f"Status: {hillcrest.client_status}")
su.log_info(f"Brand: {hillcrest.branding_config.primary_color}")
su.log_info(f"Assigned Users: {hillcrest.get_assigned_users()}")
su.log_info(f"Primary User: {hillcrest.primary_user}")

In [ ]:
# Save Hillcrest profile using modern HydraConfigManager
success = manager.save_client(hillcrest, profiles_dir=clients_dir)
su.log_info(f"Hillcrest profile saved: {success}")

# Also re-save user (now has assigned client)
manager.save_user(user, profiles_dir=users_dir)

# Verify by loading
loaded_hillcrest = manager.load_client("HILL", profiles_dir=clients_dir)
if loaded_hillcrest:
    su.log_info(f"Verified: Loaded client {loaded_hillcrest.name}")
    su.log_info(f"Client code: {loaded_hillcrest.client_code}")
    su.log_info(f"Primary color: {loaded_hillcrest.branding_config.primary_color}")
    su.log_info(f"Assigned users: {loaded_hillcrest.get_assigned_users()}")
else:
    su.log_warning("Could not load saved client profile")

## 4. Test 1Password Integration

Retrieve credentials from 1Password (if available).

In [ ]:
from siege_utilities.config import get_google_service_account_from_1password

# Check if 1Password is available
if cred_manager.available_backends.get('1password'):
    su.log_info("=== 1Password Available ===")
    
    # List stored credentials with siege-utilities tag
    stored_creds = cred_manager.list_stored_credentials()
    su.log_info(f"Found {len(stored_creds)} stored credentials:")
    for cred in stored_creds:
        su.log_info(f"  - {cred.get('service', 'unknown')} ({cred.get('backend', 'unknown')})")
else:
    su.log_info("1Password not available - skipping 1Password tests")

In [ ]:
# Try to get Google Analytics service account from 1Password
# The item title should match what's stored in 1Password

ga_service_account = None

if cred_manager.available_backends.get('1password'):
    # Try common item titles
    item_titles = [
        "Google Analytics Service Account - Multi-Client Reporter",
        "Google Analytics Service Account",
        "GA4 Service Account",
        "Hillcrest GA Service Account"
    ]
    
    for title in item_titles:
        try:
            su.log_info(f"Trying: {title}...")
            ga_service_account = get_google_service_account_from_1password(title)
            if ga_service_account:
                su.log_info("Found service account!")
                su.log_info(f"   Project: {ga_service_account.get('project_id')}")
                su.log_info(f"   Email: {ga_service_account.get('client_email')}")
                break
        except Exception as e:
            su.log_info(f"   Not found or error: {str(e)[:50]}")
            continue
    
    if not ga_service_account:
        su.log_warning("No GA service account found in 1Password")
        su.log_info("To store one, use:")
        su.log_info("  from siege_utilities.config import store_ga_service_account_from_file")
        su.log_info("  store_ga_service_account_from_file('/path/to/service-account.json')")
else:
    su.log_info("Skipping 1Password credential retrieval (not available)")

## 5. Generate Branded PDF Report

Create a sample report using Hillcrest branding.

In [ ]:
import pandas as pd
import numpy as np

# Create sample data for the report
np.random.seed(42)

# Sample polling data
poll_data = pd.DataFrame({
    'Candidate': ['Smith', 'Jones', 'Williams', 'Undecided'],
    'Support (%)': [42, 38, 12, 8],
    'Previous (%)': [40, 39, 13, 8],
    'Change': ['+2', '-1', '-1', '0']
})

# Sample demographic breakdown
demo_data = pd.DataFrame({
    'Age Group': ['18-29', '30-44', '45-59', '60+'],
    'Smith': [35, 40, 45, 48],
    'Jones': [45, 42, 35, 32],
    'Williams': [15, 12, 12, 10]
})

su.log_info("=== Sample Polling Data ===")
su.log_info(f"\n{poll_data.to_string(index=False)}")
su.log_info("=== Demographic Breakdown ===")
su.log_info(f"\n{demo_data.to_string(index=False)}")

In [ ]:
from siege_utilities.reporting.report_generator import ReportGenerator

# Initialize ReportGenerator with Hillcrest branding — output goes to OUTPUT_DIR
rg = ReportGenerator(
    client_name="Hillcrest",
    output_dir=OUTPUT_DIR
)

su.log_info("=== ReportGenerator Initialized ===")
su.log_info(f"Client: {rg.client_name}")
su.log_info(f"Output directory: {rg.output_dir}")

In [ ]:
# Build report content
report_content = {
    'sections': [],
    'metadata': {
        'title': 'Sample Polling Report',
        'subtitle': 'Q1 2026 Survey Results',
        'author': 'Siege Analytics',
        'date': datetime.now().strftime('%B %d, %Y'),
        'client': 'Hillcrest'
    }
}

# Add executive summary
executive_summary = """
This polling report presents findings from a survey of 1,200 likely voters 
conducted January 10-15, 2026. Key findings include:

- Smith leads with 42% support, up 2 points from previous poll
- Jones at 38%, down 1 point
- Williams at 12%, essentially unchanged
- 8% remain undecided with 3 weeks until election

Margin of error: +/-2.8 percentage points (95% confidence)
"""

report_content = rg.add_text_section(
    report_content, 
    'Executive Summary', 
    executive_summary
)

su.log_info("Added executive summary")

In [ ]:
# Add polling data table
report_content = rg.add_table_section(
    report_content,
    'Current Poll Results',
    poll_data
)

su.log_info("Added polling data table")

In [ ]:
# Add demographic breakdown table
report_content = rg.add_table_section(
    report_content,
    'Support by Age Group',
    demo_data
)

su.log_info("Added demographic breakdown table")

In [ ]:
# Add methodology section
methodology = """
Survey Methodology:

- Sample Size: 1,200 likely voters
- Mode: Mixed (60% cell, 40% landline)
- Field Dates: January 10-15, 2026
- Weighting: Age, gender, education, party registration
- Margin of Error: +/-2.8 percentage points

The survey was conducted by Siege Analytics using live interviewers. 
Respondents were selected using stratified random sampling from 
voter file records.
"""

report_content = rg.add_text_section(
    report_content,
    'Methodology',
    methodology
)

su.log_info("Added methodology section")

In [ ]:
# Generate the PDF
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_path = OUTPUT_DIR / f"hillcrest_poll_report_{timestamp}.pdf"

su.log_info(f"Generating PDF: {output_path}")

try:
    rg.generate_pdf_report(report_content, str(output_path))
    
    if output_path.exists():
        size_kb = output_path.stat().st_size / 1024
        su.log_info("PDF Generated Successfully!")
        su.log_info(f"   Path: {output_path}")
        su.log_info(f"   Size: {size_kb:.1f} KB")
        su.log_info(f"   Sections: {len(report_content['sections'])}")
    else:
        su.log_error("PDF was not created")
except Exception as e:
    su.log_error(f"Error generating PDF: {e}")
    import traceback
    traceback.print_exc()

## 6. Verify Profile Persistence

Verify that profiles are correctly saved and can be reloaded.

In [ ]:
# Reload and verify all profiles using modern API
su.log_info("=== Verification: Reload All Profiles ===")

# User profile
loaded_user = manager.load_user("dheeraj", profiles_dir=users_dir)
if loaded_user:
    su.log_info(f"[OK] User: {loaded_user.person_id} ({loaded_user.name})")
    su.log_info(f"   Username: {loaded_user.username}")
    su.log_info(f"   Default format: {loaded_user.default_output_format}")
    su.log_info(f"   Assigned clients: {loaded_user.get_assigned_clients()}")
else:
    su.log_error("User profile not found")

# Client profile
loaded_client = manager.load_client("HILL", profiles_dir=clients_dir)
if loaded_client:
    su.log_info(f"[OK] Client: {loaded_client.name}")
    su.log_info(f"   Code: {loaded_client.client_code}")
    su.log_info(f"   Industry: {loaded_client.industry}")
    su.log_info(f"   Primary color: {loaded_client.branding_config.primary_color}")
    su.log_info(f"   Primary font: {loaded_client.branding_config.primary_font}")
    su.log_info(f"   Assigned users: {loaded_client.get_assigned_users()}")
else:
    su.log_error("Client profile not found")

In [ ]:
# Check saved profile files
su.log_info("=== Profile Files ===")

profiles_base = Path.home() / ".siege_utilities" / "profiles"

for profile_type in ["users", "clients"]:
    profile_dir = profiles_base / profile_type
    if profile_dir.exists():
        files = list(profile_dir.glob("*.yaml"))
        su.log_info(f"{profile_type.title()}:")
        for f in files:
            size = f.stat().st_size
            su.log_info(f"  - {f.name} ({size} bytes)")
    else:
        su.log_warning(f"{profile_type.title()}: Directory not found")

## 7. Secure Shell Execution

`siege_utilities` provides a security-validated shell execution layer via the
`siege_utilities.files.shell` module. This layer wraps `subprocess.Popen` with
a command whitelist, injection-character blocking, path-traversal detection, and
sensitive-path guards. The module exposes:

| Symbol | Purpose |
|--------|---------|
| `ALLOWED_COMMANDS` | Default whitelist of safe, read-only base commands |
| `validate_command_safety()` | Validate a command string/list against the whitelist and security rules |
| `run_subprocess()` | Execute a validated command and return its stdout/stderr |
| `SecurityError` | Raised when a command fails any security check |

This section demonstrates both the **success** and **failure** paths so you can
see exactly how the guard rails work.

In [ ]:
# --- Import the secure shell execution module ---
from siege_utilities.files.shell import (
    run_subprocess,
    validate_command_safety,
    ALLOWED_COMMANDS,
    SecurityError,
)

su.log_info("=== Secure Shell Module Imported ===")
su.log_info(f"ALLOWED_COMMANDS whitelist ({len(ALLOWED_COMMANDS)} entries):")
for cmd in sorted(ALLOWED_COMMANDS):
    su.log_info(f"  - {cmd}")

In [ ]:
# --- validate_command_safety(): SUCCESS cases ---
su.log_info("=== validate_command_safety() -- Safe Commands ===")

safe_commands = [
    "ls -la",
    "echo hello world",
    "whoami",
    "date",
    ["hostname"],
]

for cmd in safe_commands:
    validated = validate_command_safety(cmd)
    su.log_info(f"  Input:     {cmd!r}")
    su.log_info(f"  Validated: {validated}")
    su.log_info("")

In [ ]:
# --- validate_command_safety(): FAILURE cases ---
# Each of these should raise SecurityError for a different reason.
su.log_info("=== validate_command_safety() -- Blocked Commands ===")

unsafe_commands = [
    ("rm -rf /",                "command not in whitelist"),
    ("ls; rm -rf /",            "injection character ';'"),
    ("echo $(cat /etc/passwd)", "injection character '$'"),
    ("cat /etc/shadow",         "sensitive path /etc/shadow"),
    ("ls ../../secrets",        "path traversal '..'"),
    ("echo hello | wall",       "injection character '|'"),
]

for cmd, reason in unsafe_commands:
    try:
        validate_command_safety(cmd)
        su.log_error(f"  UNEXPECTED PASS: {cmd!r} (expected block: {reason})")
    except SecurityError as exc:
        su.log_info(f"  [BLOCKED] {cmd!r}")
        su.log_info(f"    Reason: {exc}")
    except ValueError as exc:
        su.log_info(f"  [BLOCKED] {cmd!r}")
        su.log_info(f"    ValueError: {exc}")
    su.log_info("")

In [ ]:
# --- run_subprocess(): SUCCESS case ---
su.log_info("=== run_subprocess() -- Safe Execution ===")

output = run_subprocess("echo hello from siege_utilities")
su.log_info(f"  echo output: {output.strip()!r}")

output = run_subprocess("ls -la")
su.log_info(f"  ls output (first 5 lines):")
for line in output.strip().splitlines()[:5]:
    su.log_info(f"    {line}")

output = run_subprocess("whoami")
su.log_info(f"  whoami: {output.strip()!r}")

output = run_subprocess("date")
su.log_info(f"  date: {output.strip()!r}")

In [ ]:
# --- run_subprocess(): FAILURE case ---
# Attempting to run a blocked command should raise SecurityError.
su.log_info("=== run_subprocess() -- Blocked Execution ===")

blocked_commands = [
    "rm -rf /",
    "python -c 'import os; os.system(\"id\")'",
    "curl http://example.com",
]

for cmd in blocked_commands:
    try:
        run_subprocess(cmd)
        su.log_error(f"  UNEXPECTED PASS: {cmd!r}")
    except SecurityError as exc:
        su.log_info(f"  [BLOCKED] {cmd!r}")
        su.log_info(f"    SecurityError: {exc}")
    except ValueError as exc:
        su.log_info(f"  [BLOCKED] {cmd!r}")
        su.log_info(f"    ValueError: {exc}")
    su.log_info("")

In [ ]:
# --- run_subprocess() with a custom allow_list ---
# Demonstrate extending the whitelist for a specific use case.
su.log_info("=== run_subprocess() -- Custom Allow List ===")

custom_allow = ALLOWED_COMMANDS | {"python3"}  # extend the default set
su.log_info(f"  Custom whitelist adds: python3")

# This would normally be blocked but succeeds with the custom list
# Note: we use --version instead of -c because the security layer
# blocks parentheses/special characters in arguments.
output = run_subprocess(
    ["python3", "--version"],
    allow_list=custom_allow,
)
su.log_info(f"  python3 output: {output.strip()!r}")

## 8. Summary

Test results summary.

In [ ]:
# Summary of all tests
su.log_info("="*50)
su.log_info("PROFILE & BRANDING SYSTEM TEST RESULTS")
su.log_info("="*50)

# Quick shell-execution smoke test for the summary
_shell_ok = False
try:
    _val = validate_command_safety("echo ok")
    _out = run_subprocess("echo ok")
    _shell_ok = _out.strip() == "ok"
except Exception:
    pass

results = {
    "Credential Backends": "1password" in [k for k,v in cred_manager.available_backends.items() if v],
    "User Profile Created": manager.load_user("dheeraj", profiles_dir=users_dir) is not None,
    "Client Profile Created": manager.load_client("HILL", profiles_dir=clients_dir) is not None,
    "1Password Integration": cred_manager.available_backends.get('1password', False),
    "PDF Generation": output_path.exists() if 'output_path' in dir() else False,
    "Secure Shell Execution": _shell_ok,
}

for test, passed in results.items():
    symbol = "[PASS]" if passed else "[FAIL]"
    su.log_info(f"{symbol} {test}")

passed = sum(results.values())
total = len(results)
su.log_info(f"{passed}/{total} tests passed")

if passed == total:
    su.log_info("All Profile/Branding tests passed!")
else:
    su.log_warning("Some tests need attention")

# Clean up HydraConfigManager
manager.cleanup()

## Next Steps

If all tests pass:

1. **Update Hillcrest branding colors** - Edit the BrandingConfig with actual Hillcrest colors
2. **Add logo** - Set `logo_path` to actual Hillcrest logo file
3. **Store GA credentials** - If you have GA service account, store it:
   ```python
   from siege_utilities.config import store_ga_service_account_from_file
   store_ga_service_account_from_file('/path/to/service-account.json', 
                                       item_title='Hillcrest GA Service Account')
   ```
4. **Test with real data** - Use actual polling data from Hillcrest projects